In [ ]:
import os
import sympy
import pickle
import numpy as np
import pandas as pd
from PIL import Image
import word2image as w2i
from skimage import color
import matplotlib.pyplot as plt
from scipy.stats import kstest, norm

In [ ]:
def calculate_ratio_of_value(elements, target_value=100.0):
    count_target_value = elements.count(target_value)
    total_elements = len(elements)
    if total_elements > 0:
        ratio = count_target_value / total_elements
    else:
        ratio = 0
    return ratio

In [ ]:
def leave_first_element(List):
    lst=[]
    for x in List:
        lst.append(x[0])
    return lst

In [ ]:
def extractColors(base_dir, search_term=""):
    filename = search_term + ".png"

    resize_image_dir = base_dir

    img_url = resize_image_dir + '/' + filename

    # Open the image file
    image = Image.open(img_url)

    # Get the size of the image in pixels
    width, height = image.size

    # Create an empty list to store the RGB values for each pixel
    rgb_list = []
    lab_list=[]
    # Loop over each pixel in the image and extract the RGB values
    for y in range(height):
        for x in range(width):
            pixel = image.getpixel((x, y))
            if isinstance(pixel, int):
                r, g, b = pixel, pixel, pixel
            else:
                r, g, b = pixel[:3]
            rgb_list.append((r, g, b))

    for (r, g, b) in rgb_list:
        if isinstance(r, int) and isinstance(g, int) and isinstance(b, int) and 0 <= r <= 255 and 0 <= g <= 255 and 0 <= b <= 255:
            # Convert RGB to XYZ
            r_norm = r / 255
            g_norm = g / 255
            b_norm = b / 255

            r_lin = r_norm if r_norm <= 0.04045 else ((r_norm + 0.055) / 1.055) ** 2.4
            g_lin = g_norm if g_norm <= 0.04045 else ((g_norm + 0.055) / 1.055) ** 2.4
            b_lin = b_norm if b_norm <= 0.04045 else ((b_norm + 0.055) / 1.055) ** 2.4

            x = r_lin * 0.4124564 + g_lin * 0.3575761 + b_lin * 0.1804375
            y = r_lin * 0.2126729 + g_lin * 0.7151522 + b_lin * 0.0721750
            z = r_lin * 0.0193339 + g_lin * 0.1191920 + b_lin * 0.9503041

            # Convert XYZ to Lab
            xn = 0.95047
            yn = 1.00000
            zn = 1.08883

            f_x = x ** (1/3) if x > 0.008856 else 7.787 * x + 16 / 116
            f_y = y ** (1/3) if y > 0.008856 else 7.787 * y + 16 / 116
            f_z = z ** (1/3) if z > 0.008856 else 7.787 * z + 16 / 116

            L = 116 * f_y - 16
            a = 500 * (f_x - f_y)
            b = 200 * (f_y - f_z)

            lab_list.append((L, a, b))
        else:
            print("Invalid RGB value:", (r, g, b))

    if not lab_list:
        return None

    # Convert CIELAB values to CIELCH values
    lch_list = []

    for lab in lab_list:
        L, a, b = lab
        C = np.sqrt(a**2 + b**2)
        H = np.degrees(np.arctan2(b, a))

        lch_list.append((L, C, H))

    # Create Dataframe
    df = pd.DataFrame({'cielab': lab_list, 'cielch': lch_list})
    df['occurrence'] = df.groupby('cielab')['cielab'].transform('count')
    df['ratios'] = df["occurrence"]/df["occurrence"].sum()



    return df["cielch"]

In [ ]:
# import nltk
# import random
# from nltk.corpus import wordnet as wn

# # # Run this if you haven't already downloaded WordNet
# # nltk.download('wordnet')
# # nltk.download('omw-1.4')

# # Get all synsets (regardless of part of speech)
# all_synsets = wn.all_synsets()

# # Extract word names from synsets
# all_words = {x.name().split('.', 1)[0] for x in all_synsets}

# # Keep only clean, single-token words (no space, underscore, hyphen)
# clean_words = [word for word in all_words if ' ' not in word and '_' not in word and '-' not in word]

# # Randomly select up to 100 words
# wild_list = random.sample(clean_words, min(100, len(clean_words)))

In [ ]:
wild_list = [
    'hemerocallis', 'archiannelid', 'ponce', 'malvales', 'wegener', 'seraphic', 'heroics', 
    'cordaites', 'taxaceae', 'coerebidae', 'seeder', 'leptotyphlopidae', 'entolomataceae', 
    'disclosure', 'psittacosis', 'spectroscope', 'explode', 'klyuchevskaya', 'scaled', 'transfiguration',
    'dingo', 'megadeath', 'commissar', 'monotone', 'cart', 'bugbear', 'parsonage', 'bathtub', 'earmuff', 
    'niqaabi', 'snowshoe', 'cuculiformes', 'chelicerae', 'aardvark', 'belemnitic', 'toby', 'enantiomorph', 
    'videocassette', 'psychologist', 'keratin', 'amphineura', 'orchid', 'baccarat', 'czarina', 'mitterrand',
    'mariner', 'kosteletzya', 'kutuzov', 'file', 'caring', 'unilateralism', 'latrodectus', 'hathaway', 
    'babylonian', 'hasdrubal', 'chorally', 'polygonum', 'festival', 'nectarine', 'clavichord', 'geophyte',
    'prowl', 'mint', 'competitive', 'mergus', 'cryptogramma', 'kilroy', 'pinkify', 'burner', 'watchtower', 
    'hurrah', 'tambour', 'palaemonidae', 'girder', 'excited', 'tumble', 'noisy', 'conima', 'aerophile', 
    'haley', 'tread', 'immigration', 'pontifex', 'timbrel', 'studbook', 'erring', 'cathedral', 'porringer', 
    'helodermatidae', 'hakham', 'proserpina', 'asteroid', 'declutch', 'isohel', 'carcinoma', 'boy', 
    'saprophytic', 'lupinus', 'malacostraca', 'tank'
]

In [ ]:
for wild_word in wild_list:
    w2i.word2image(wild_word,10000,original_dir="./Images/original_images", temp_dir="./Images/temp", resize_dir="./Images/resize_images", check_text=True)

In [ ]:
import os
import pickle

# Directory containing the resized images
base_dir = './Images/resize_images'

# Path to save the processed data file
save_file = '/content/drive/MyDrive/wild type 100/processed_data.pkl'

# Load previously saved data
processed_data = load_processed_data(save_file)

# Only include wild_list items that actually exist as PNG files in base_dir
files = [f"{word}.png" for word in wild_list if os.path.exists(os.path.join(base_dir, f"{word}.png"))]
print(f"Found {len(files)} matching PNG files.")

# Process each valid image
for file_name in files:
    search_term = os.path.splitext(file_name)[0]

    if search_term in processed_data:
        print(f"Skipping {file_name}, already processed.")
        continue

    try:
        df_cielch = extractColors(base_dir, search_term)
        if df_cielch is None:
            print(f"No data extracted for {file_name}.")
            continue
    except Exception as e:
        print(f"Error processing {file_name}: {e}")
        continue

    first_elements = leave_first_element(df_cielch)
    ratio = calculate_ratio_of_value(first_elements, target_value=100.00)

    print(file_name)
    print(ratio)

    processed_data[search_term] = ratio
    print(f"Processed {file_name}: Ratio = {ratio}")

    save_processed_data(processed_data, save_file)

print("Processing complete!")


In [ ]:
# Path to the saved data file. Allows using previously saved values 
# without reprocessing the original 100 screenshots.
save_file = './processed_data.pkl'

# Function to load saved processed data
def load_processed_data(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            return pickle.load(f)
    return {}

# Load the processed data
processed_data = load_processed_data(save_file)


In [ ]:
# Extract ratio values
values = np.array(list(processed_data.values()))

# Calculate the mean and standard deviation
mean_value = np.mean(values)  # Mean of the ratio values
std_dev = np.std(values)      # Standard deviation of the ratio values

In [ ]:
# Normality test for white grid

# Perform KS test
# Data normalization (based on mean and standard deviation)
normalized_data = (values - mean_value) / std_dev
statistic, p_value = kstest(normalized_data, 'norm')

print("Kolmogorov-Smirnov Test Results:")
print(f"Test Statistic = {statistic:.4f}, p-value = {p_value:.4f}")

# Interpret the results
if p_value > 0.05:
    print("Fail to reject the null hypothesis: The data follows a normal distribution.")
else:
    print("Reject the null hypothesis: The data does not follow a normal distribution.")


In [ ]:
# Distribution analysis of white grid ratio for 100 random words

print(f"Number of data points: {len(values)}")
print(f"Mean: {mean_value:.4f}")
print(f"Standard Deviation: {std_dev:.4f}")

# Plot histogram and normal distribution curve
plt.figure(figsize=(3.5, 2.5))

# Calculate ±1 standard deviation boundaries
std_dev1_left = mean_value - std_dev
std_dev1_right = mean_value + std_dev

# Calculate ±2 standard deviation boundaries
std_dev2_left = mean_value - 2 * std_dev
std_dev2_right = mean_value + 2 * std_dev

# Plot histogram
count, bins, ignored = plt.hist(values, bins=100, range=(0.27, 0.37),
                                 color='lightgrey', edgecolor='black', density=False)

# Add mean and standard deviation lines
plt.axvline(mean_value, color='black', linestyle='dashed', linewidth=1, label=f"Mean: {mean_value:.4f}")
plt.axvline(std_dev1_left, color='blue', linestyle='dotted', linewidth=1, label=f"±1 Std Dev: {std_dev:.4f}")
plt.axvline(std_dev1_right, color='blue', linestyle='dotted', linewidth=1)
plt.axvline(std_dev2_left, color='skyblue', linestyle='dotted', linewidth=1, label=f"±2 Std Dev: {std_dev*2:.4f}")
plt.axvline(std_dev2_right, color='skyblue', linestyle='dotted', linewidth=1)

# Configure plot settings
plt.xlim(0.27, 0.37)  # Set x-axis range
plt.ylim(0, 16)
plt.yticks([5, 10, 15])
plt.xlabel(r'Ratio of $L=100.00$ Pixels', fontsize=8)
plt.ylabel('Frequency', fontsize=8)
plt.legend(fontsize=7, loc='upper right')

# Save the plot
save_path = '/content/drive/MyDrive/wild_type_histogram.png'
plt.savefig(save_path, format='png', dpi=300, bbox_inches='tight')  # High resolution and tight layout
print(f"Graph has been saved to: {save_path}")

plt.show()
plt.close()